# ASON: Adaptive Spectral Operator Network for HSI-MSI Fusion

Fully self-contained notebook. Datasets: **Chikusei** (128 bands) + **CAVE** (31 bands).
Protocol: x4 Wald (Gaussian blur sigma=1.2, 3-band Gaussian SRF).

## BEFORE RUNNING: Select GPU T4 x2
1. Click **Notebook menu** (top-right `...`)
2. Go to **Settings** -> **Accelerator**
3. Select **GPU T4 x2** (NOT "None" or "P100")
4. Then click **Run All**

P100 (sm_60) is incompatible with PyTorch >= 2.6. T4 (sm_75) is ~10x faster than CPU.

## 1. Environment

In [ ]:
import os, sys, json, time, math, warnings, glob
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
print(f'python {sys.version.split()[0]}  torch {torch.__version__}  numpy {np.__version__}')

DEVICE = 'cpu'
GPU_OK = False
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    print(f'gpu     {p.name} {p.total_memory/2**30:.1f} GB {arch}')
    print(f'built   {" ".join(built)}')
    GPU_OK = arch in built
    if GPU_OK:
        DEVICE = 'cuda'
        torch.backends.cudnn.benchmark = True
        print(f'amp     {"fp16 with tensor cores" if p.major >= 7 else "fp16"}')
    else:
        print(f'\n*** WARNING: {p.name} ({arch}) is NOT supported by this PyTorch build.')
        print(f'*** Training will run on CPU (slow but functional).')
        print(f'*** FIX: Notebook menu -> Settings -> Accelerator -> "GPU T4 x2"')
else:
    print('no GPU detected - training on CPU')

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK)
print(f'workdir {os.getcwd()}  device={DEVICE}')
!pip install -q h5py

if DEVICE != 'cuda':
    raise SystemExit(
        '\n*** ABORT: No compatible GPU detected. ***\n'
        '*** Go to Notebook menu -> Settings -> Accelerator -> GPU T4 x2 ***\n'
        '*** Then re-run this notebook. ***'
    )
print("GPU ready - proceeding with training")

## 2. Null-space projector (self-contained)

In [ ]:
%%writefile nullspace.py
r"""Range/null-space decomposition and the observed-operator projector.

THE FORMULATION
---------------
The low-resolution hyperspectral observation fixes part of the solution:

    X = D(Y),     D = blur then decimate,

and D has a non-trivial null space: many HR cubes explain the same LR cube
exactly.  Writing the split

    Y_hat = D_pinv(X) + P_perp( v )
            \_______/    \_______/
           fixed by        genuinely free
           the data       (what a fusion
                          method must supply)

with D_pinv = D^T (D D^T)^-1 and P_perp = I - D_pinv D, the reconstruction
satisfies

    D(Y_hat) = D(D_pinv X) + D(P_perp v) = X + 0 = X

for *any* v whatsoever.  Data consistency is an algebraic identity, not a
penalty.

WHAT THIS MEANS FOR A GENERATIVE METHOD
---------------------------------------
SpectralFlow never asks a network to predict the whole cube.  The range
component D_pinv(X) is computed in closed form from the measured LR-HSI.  The
generative model only ever produces the null-space component - the degrees of
freedom the observation genuinely leaves free - and the sampler re-projects
onto the consistent set at every reverse step.  This is what separates the
formulation from a diffusion method that generates freely and hopes the
observation is satisfied.

The cost of the projection is one small CG solve in *low-resolution* space per
call.  The inner system (D D^T + ridge I) lives on the LR grid, which is small,
so the projector runs inside every reverse step without dominating the cost.

The operator D may be fixed (a known eval kernel) or estimated by the network
(blind setting).  The identity D P_perp = 0 holds with respect to whichever D
is supplied; estimating the kernel and building the projector from it couples
the two, which is what makes the degradation head load-bearing.
"""

from __future__ import annotations

from typing import Callable, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------- operators
def _kernel_weight(kernel: torch.Tensor, channels: int, ksize: int,
                   batch: int) -> torch.Tensor:
    """Expand a per-sample kernel into grouped-conv weights."""
    k = kernel.reshape(batch, 1, 1, ksize, ksize)
    return k.expand(batch, channels, 1, ksize, ksize).reshape(
        batch * channels, 1, ksize, ksize)


class DegradationOperator(nn.Module):
    """D and its exact adjoint D^T.

    Zero padding throughout: reflect padding is not self-adjoint, and mixing it
    with a transposed-convolution adjoint silently breaks <Dx,y> = <x,D^T y>,
    which would make every CG solve here converge to the wrong answer while
    still looking healthy.
    """

    def __init__(self, scale: int, ksize: int = 9, sigma: float = 1.2):
        super().__init__()
        self.scale, self.ksize = scale, ksize
        ax = torch.arange(ksize, dtype=torch.float32) - (ksize - 1) / 2
        g = torch.exp(-0.5 * (ax / sigma) ** 2)
        k = torch.outer(g, g)
        self.register_buffer("default_kernel", (k / k.sum())[None])

    def _k(self, kernel: Optional[torch.Tensor], batch: int,
           device, dtype) -> torch.Tensor:
        k = self.default_kernel if kernel is None else kernel
        k = k.to(device=device, dtype=dtype)
        if k.dim() == 2:
            k = k[None]
        if k.shape[0] == 1 and batch > 1:
            k = k.expand(batch, -1, -1)
        return k.reshape(batch, self.ksize, self.ksize)

    def forward(self, y: torch.Tensor, kernel: Optional[torch.Tensor] = None
                ) -> torch.Tensor:
        """D: [B,C,H,W] -> [B,C,H/s,W/s]."""
        b, c, h, w = y.shape
        k = self._k(kernel, b, y.device, y.dtype)
        wgt = _kernel_weight(k, c, self.ksize, b)
        pad = self.ksize // 2
        yr = F.pad(y.reshape(1, b * c, h, w), (pad,) * 4, mode="constant")
        out = F.conv2d(yr, wgt, groups=b * c).reshape(b, c, h, w)
        return out[..., ::self.scale, ::self.scale].contiguous()

    def transpose(self, x: torch.Tensor, out_hw: Tuple[int, int],
                  kernel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """D^T: [B,C,h,w] -> [B,C,H,W]. Zero-insert, then correlate with the
        flipped kernel."""
        b, c, h, w = x.shape
        up = x.new_zeros(b, c, out_hw[0], out_hw[1])
        up[..., ::self.scale, ::self.scale] = x
        k = self._k(kernel, b, x.device, x.dtype)
        k = torch.flip(k, dims=(-2, -1))
        wgt = _kernel_weight(k, c, self.ksize, b)
        pad = self.ksize // 2
        upr = F.pad(up.reshape(1, b * c, *out_hw), (pad,) * 4, mode="constant")
        return F.conv2d(upr, wgt, groups=b * c).reshape(b, c, *out_hw)


# ------------------------------------------------------------------ solver
def conjugate_gradient(applyA: Callable[[torch.Tensor], torch.Tensor],
                       rhs: torch.Tensor, steps: int,
                       tol: float = 1e-10) -> torch.Tensor:
    """Batched CG for a symmetric positive-definite operator.

    Per-sample scalars, so one easy sample in a batch cannot stall while a hard
    one converges (or vice versa).
    """
    z = torch.zeros_like(rhs)
    r = rhs - applyA(z)
    p = r.clone()
    rs = (r * r).flatten(1).sum(1)
    for _ in range(steps):
        ap = applyA(p)
        denom = (p * ap).flatten(1).sum(1)
        alpha = (rs / denom.clamp_min(tol)).reshape(-1, 1, 1, 1)
        z = z + alpha * p
        r = r - alpha * ap
        rs_new = (r * r).flatten(1).sum(1)
        beta = (rs_new / rs.clamp_min(tol)).reshape(-1, 1, 1, 1)
        p = r + beta * p
        rs = rs_new
    return z


# --------------------------------------------------------------- projector
class RangeNullProjector(nn.Module):
    """Computes D_pinv(x) and P_perp(v) for the blur-and-decimate operator.

    Both need the same inner solve, (D D^T + ridge I) z = rhs, taken in
    low-resolution space where the system is small.  Defaults are the ones
    measured in proposal1/docs: 8 CG steps put the identity at solver
    tolerance, and ridge=1e-6 keeps the solve well conditioned at negligible
    cost to the exactness claim.
    """

    def __init__(self, scale: int, ksize: int = 9, sigma: float = 1.2,
                 cg_steps: int = 8, ridge: float = 1e-6):
        super().__init__()
        self.D = DegradationOperator(scale, ksize, sigma)
        self.scale, self.cg_steps, self.ridge = scale, cg_steps, ridge

    def _normal_op(self, out_hw: Tuple[int, int],
                   kernel: Optional[torch.Tensor]) -> Callable:
        """z -> (D D^T + ridge I) z, acting in low-resolution space."""
        def applyA(z: torch.Tensor) -> torch.Tensor:
            return self.D(self.D.transpose(z, out_hw, kernel), kernel) \
                + self.ridge * z
        return applyA

    def pinv(self, x_lr: torch.Tensor, out_hw: Tuple[int, int],
             kernel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """D_pinv x = D^T (D D^T)^-1 x - the component the data determines."""
        z = conjugate_gradient(self._normal_op(out_hw, kernel), x_lr,
                               self.cg_steps)
        return self.D.transpose(z, out_hw, kernel)

    def project_null(self, v: torch.Tensor,
                     kernel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """P_perp v = v - D_pinv(D v) - the component the data cannot see."""
        hw = (v.shape[-2], v.shape[-1])
        return v - self.pinv(self.D(v, kernel), hw, kernel)

    def consistent(self, x_lr: torch.Tensor, v: torch.Tensor,
                   kernel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """D_pinv(x) + P_perp(v): the consistent set the sampler lives on."""
        hw = (v.shape[-2], v.shape[-1])
        return self.pinv(x_lr, hw, kernel) + self.project_null(v, kernel)


# ------------------------------------------------------------- degradation
def decode_degradation_params(raw: torch.Tensor, min_sigma: float = 0.3
                              ) -> torch.Tensor:
    """Convert unconstrained head outputs into physical degradation parameters.

    Returns physical ``[sigma_x, sigma_y, sin(2 theta), cos(2 theta), noise]``
    values.  The doubled angle avoids the half-turn ambiguity of an ellipse.
    """
    if raw.ndim != 2 or raw.shape[1] < 5:
        raise ValueError("expected raw degradation parameters [B, >=5]")
    sigma_x = F.softplus(raw[:, 0]) + min_sigma
    sigma_y = F.softplus(raw[:, 1]) + min_sigma
    orient = raw[:, 2:4]
    orient_norm = orient.norm(dim=1, keepdim=True)
    orient_unit = orient / orient_norm.clamp_min(1e-6)
    default_orient = torch.zeros_like(orient)
    default_orient[:, 1] = 1.0
    orient = torch.where(orient_norm > 1e-6, orient_unit, default_orient)
    noise = F.softplus(raw[:, 4])
    return torch.cat([sigma_x[:, None], sigma_y[:, None], orient, noise[:, None]], dim=1)


def kernel_from_params(params: torch.Tensor, ksize: int = 9,
                       min_sigma: float = 0.3) -> torch.Tensor:
    """Build a differentiable anisotropic Gaussian kernel from *physical*
    degradation parameters (sx, sy, sin2t, cos2t, ...).  ``params`` must have
    passed through :func:`decode_degradation_params`."""
    sx = params[:, 0].clamp_min(min_sigma)
    sy = params[:, 1].clamp_min(min_sigma)
    s2, c2 = params[:, 2], params[:, 3]
    norm = torch.sqrt(s2 ** 2 + c2 ** 2).clamp_min(1e-6)
    theta = 0.5 * torch.atan2(s2 / norm, c2 / norm)

    ax = torch.arange(ksize, device=params.device, dtype=params.dtype)
    ax = ax - (ksize - 1) / 2
    yy, xx = torch.meshgrid(ax, ax, indexing="ij")
    xx, yy = xx[None], yy[None]                      # [1,k,k]
    cos_t = torch.cos(theta)[:, None, None]
    sin_t = torch.sin(theta)[:, None, None]
    xr = xx * cos_t + yy * sin_t
    yr = -xx * sin_t + yy * cos_t
    k = torch.exp(-0.5 * ((xr / sx[:, None, None]) ** 2
                          + (yr / sy[:, None, None]) ** 2))
    return k / k.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)


# ------------------------------------------------------------- verification
@torch.no_grad()
def check_adjoint(scale: int = 4, ksize: int = 9, size: int = 32,
                  channels: int = 5, tol: float = 1e-5) -> float:
    """<D y, x> must equal <y, D^T x>."""
    torch.manual_seed(0)
    D = DegradationOperator(scale, ksize)
    y = torch.randn(2, channels, size, size)
    x = torch.randn(2, channels, size // scale, size // scale)
    lhs = (D(y) * x).sum()
    rhs = (y * D.transpose(x, (size, size))).sum()
    err = abs((lhs - rhs).item()) / max(abs(lhs.item()), 1e-12)
    print(f"[check] adjoint <Dy,x> vs <y,D^Tx>: rel err {err:.2e} "
          f"({'PASS' if err < tol else 'FAIL'})")
    return err


@torch.no_grad()
def check_consistency(scale: int = 4, size: int = 32, channels: int = 5,
                      tol: float = 1e-3) -> float:
    """THE claim: D(consistent(x, v)) == x for an arbitrary v."""
    torch.manual_seed(0)
    P = RangeNullProjector(scale, cg_steps=30, ridge=0.0)
    x = torch.rand(2, channels, size // scale, size // scale)
    v = torch.randn(2, channels, size, size) * 3.0      # arbitrary, large
    out = P.consistent(x, v)
    err = (P.D(out) - x).abs().max().item() / max(x.abs().max().item(), 1e-12)
    print(f"[check] data consistency  max|D(Y_hat) - X| / max|X| = {err:.2e} "
          f"({'PASS' if err < tol else 'FAIL'})")
    return err


def run_all(verbose: bool = True) -> bool:
    ok = True
    ok &= check_adjoint() < 1e-5
    ok &= check_consistency() < 1e-3
    print(f"\n[nullspace] {'ALL PASS' if ok else 'FAILURES PRESENT'}")
    return bool(ok)


if __name__ == "__main__":
    run_all()

## 3. ASON model + loss

In [ ]:
class VelocityNet(nn.Module):
    def __init__(self, bands, msi_bands, code_dim=64, hidden=32, blocks=3):
        super().__init__()
        self.proj_m = nn.Conv2d(msi_bands, bands, 1)
        self.code_proj = nn.Linear(code_dim, hidden * 2)
        ch = 2 * bands + 1
        body = [nn.Conv2d(ch, hidden, 3, padding=1)]
        for _ in range(blocks - 1):
            body += [nn.SiLU(), nn.Conv2d(hidden, hidden, 3, padding=1)]
        body += [nn.SiLU(), nn.Conv2d(hidden, bands, 3, padding=1)]
        self.net = nn.Sequential(*body)
        self.film_gamma = nn.Linear(hidden * 2, hidden)
        self.film_beta = nn.Linear(hidden * 2, hidden)
        nn.init.zeros_(self.film_gamma.weight); nn.init.zeros_(self.film_gamma.bias)
        nn.init.zeros_(self.film_beta.weight); nn.init.zeros_(self.film_beta.bias)

    def forward(self, y, m, t, code):
        B, C, H, W = y.shape
        if not torch.is_tensor(t):
            t = torch.full((B,), float(t), device=y.device, dtype=y.dtype)
        tch = t.reshape(B,1,1,1).expand(B,1,H,W)
        inp = torch.cat([y, self.proj_m(m), tch], dim=1)
        code_feat = self.code_proj(code)
        gamma = self.film_gamma(code_feat).unsqueeze(-1).unsqueeze(-1)
        beta = self.film_beta(code_feat).unsqueeze(-1).unsqueeze(-1)
        x = self.net[0](inp)
        x = self.net[1](x)
        x = x * (1 + gamma) + beta
        for layer in self.net[2:]:
            x = layer(x)
        return x


class DegradationCode(nn.Module):
    def __init__(self, bands, msi_bands, code_dim=64):
        super().__init__()
        self.enc_lr = nn.Sequential(
            nn.Conv2d(bands, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(4),
            nn.Conv2d(32, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.enc_ms = nn.Sequential(
            nn.Conv2d(msi_bands, 16, 3, padding=1), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(48, 32), nn.ReLU(), nn.Linear(32, code_dim))

    def forward(self, lr_hsi, msi):
        return self.head(torch.cat([self.enc_lr(lr_hsi), self.enc_ms(msi)], 1))


class ASONNet(nn.Module):
    def __init__(self, bands, msi_bands, scale=4, hidden=32, n_blocks=3,
                 sample_steps=8, cg_steps=8, eval_sigma=1.2):
        super().__init__()
        self.bands = bands
        self.msi_bands = msi_bands
        self.scale = scale
        self.sample_steps = sample_steps
        from nullspace import RangeNullProjector
        self.projector = RangeNullProjector(scale, 9, eval_sigma, cg_steps, 1e-6)
        self.deg_code = DegradationCode(bands, msi_bands)
        self.velocity = VelocityNet(bands, msi_bands, 64, hidden, n_blocks)
        self.register_buffer("srf", torch.zeros(bands, msi_bands))

    def set_srf(self, srf):
        s = srf if srf.shape[0] == self.bands else srf.t().contiguous()
        self.srf.data = s.float()

    def pinv(self, x_lr, hw, kernel=None):
        return self.projector.pinv(x_lr, hw, kernel)

    def velocity_field(self, y, m, t, code, kernel=None):
        v = self.velocity(y, m, t, code)
        v = self.projector.project_null(v, kernel)
        return v

    def forward(self, lr_hsi, msi, kernel=None):
        hw = (msi.shape[-2], msi.shape[-1])
        code = self.deg_code(lr_hsi, msi)
        out = self.sample(lr_hsi, msi, kernel=kernel, code=code)
        return {"out": out}

    def training_step(self, x_lr, m, gt, t, kernel=None):
        hw = (gt.shape[-2], gt.shape[-1])
        code = self.deg_code(x_lr, m)
        y0 = self.pinv(x_lr, hw, kernel)
        u = gt - y0
        yt = y0 + t.view(-1,1,1,1) * u
        v = self.velocity_field(yt, m, t, code, kernel)
        return {"y0": y0, "target": u, "velocity": v, "t": t, "code": code}

    @torch.no_grad()
    def sample(self, x_lr, m, kernel=None, steps=None, code=None):
        steps = steps or self.sample_steps
        hw = (x_lr.shape[-2]*self.scale, x_lr.shape[-1]*self.scale)
        if code is None:
            code = self.deg_code(x_lr, m)
        y = self.pinv(x_lr, hw, kernel)
        for k in range(steps):
            t = (k + 0.5) / steps
            v = self.velocity(y, m, t, code)
            v = self.projector.project_null(v, kernel)
            y = y + v / steps
        return y.clamp(0, 1)


def charbonnier(x, y, eps=1e-3):
    return torch.mean(torch.sqrt((x - y)**2 + eps**2))

def sam_loss_fn(pred, target):
    p = pred.reshape(pred.shape[0], pred.shape[1], -1)
    t = target.reshape(target.shape[0], target.shape[1], -1)
    p = p / (p.norm(dim=2, keepdim=True) + 1e-8)
    t = t / (t.norm(dim=2, keepdim=True) + 1e-8)
    return torch.mean(torch.acos((p*t).sum(dim=2).clamp(-1,1)))

print("ASON model + loss defined")

## 4. Metrics

In [ ]:
def calc_psnr(pred, target, data_range=1.0):
    mse = torch.mean((pred - target)**2).item()
    return 20 * math.log10(data_range / math.sqrt(mse)) if mse > 0 else 100.0

def calc_ssim(pred, target, win=11):
    ssims = []
    for b in range(pred.shape[0]):
        C1, C2 = 0.0001, 0.0009
        mu_x = F.avg_pool2d(pred[b:b+1].unsqueeze(0), win, 1, padding=win//2).squeeze()
        mu_y = F.avg_pool2d(target[b:b+1].unsqueeze(0), win, 1, padding=win//2).squeeze()
        sig_x = F.avg_pool2d(pred[b:b+1].unsqueeze(0)**2, win, 1, padding=win//2).squeeze() - mu_x**2
        sig_y = F.avg_pool2d(target[b:b+1].unsqueeze(0)**2, win, 1, padding=win//2).squeeze() - mu_y**2
        sig_xy = F.avg_pool2d((pred[b]*target[b]).unsqueeze(0).unsqueeze(0), win, 1, padding=win//2).squeeze() - mu_x*mu_y
        ssims.append(((2*mu_x*mu_y+C1)*(2*sig_xy+C2)/((mu_x**2+mu_y**2+C1)*(sig_x+sig_y+C2))).mean().item())
    return np.mean(ssims)

def calc_sam(pred, target):
    p = pred.reshape(pred.shape[0], -1).T
    t = target.reshape(target.shape[0], -1).T
    mask = (t.norm(dim=1) > 1e-8) & (p.norm(dim=1) > 1e-8)
    if not mask.any(): return 0.0
    pn = p[mask] / (p[mask].norm(dim=1, keepdim=True) + 1e-8)
    tn = t[mask] / (t[mask].norm(dim=1, keepdim=True) + 1e-8)
    return torch.acos((pn*tn).sum(dim=1).clamp(-1,1)).mean().item() * 180 / math.pi

def calc_ergas(pred, target, scale=4):
    rmse = torch.sqrt(torch.mean((pred - target)**2, dim=(1,2)))
    mean_b = torch.mean(target, dim=(1,2)) + 1e-8
    return (100/scale * torch.mean((rmse/mean_b)**2)).item()

def all_metrics(pred, target, scale=4):
    return {"PSNR": calc_psnr(pred, target), "SSIM": calc_ssim(pred, target),
            "SAM": calc_sam(pred, target), "ERGAS": calc_ergas(pred, target, scale)}
print("Metrics defined")

## 5. Dataset loading

In [ ]:
import scipy.io as sio

def _extract_numeric(arr):
    """Recursively extract numeric data from structured/nested .mat arrays."""
    if isinstance(arr, np.ndarray):
        if arr.dtype.names:
            for name in ("gt", "data", "hsi", "image", "X"):
                if name in arr.dtype.names:
                    return _extract_numeric(arr[name])
            return _extract_numeric(arr[arr.dtype.names[0]])
        elif arr.dtype.kind == 'O':
            item = arr.flat[0] if arr.size == 1 else arr
            if isinstance(item, np.ndarray):
                return _extract_numeric(item)
            return np.asarray(item, dtype=np.float32)
        elif arr.ndim >= 2:
            return arr.astype(np.float32)
    elif isinstance(arr, np.void):
        for name in ("gt", "data", "hsi", "image", "X"):
            if name in arr.dtype.names:
                return _extract_numeric(arr[name])
        if arr.dtype.names:
            return _extract_numeric(arr[arr.dtype.names[0]])
    return np.asarray(arr, dtype=np.float32)

def load_mat(path):
    """Load .mat file (v5 via scipy, v7.3 via h5py)."""
    try:
        mat = sio.loadmat(path)
        for k, v in mat.items():
            if k.startswith("__"):
                continue
            try:
                arr = _extract_numeric(np.asarray(v))
                if arr.ndim >= 2:
                    return arr
            except Exception:
                continue
    except NotImplementedError:
        pass
    # Fallback: try h5py for v7.3 files
    try:
        import h5py
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                if k.startswith("__"):
                    continue
                try:
                    data = np.array(f[k])
                    # h5py returns transposed arrays for MATLAB
                    if data.ndim >= 2:
                        if data.shape[0] < data.shape[-1]:
                            data = np.transpose(data, (*range(1, data.ndim), 0))
                        return data.astype(np.float32)
                    # Try nested group
                    if hasattr(f[k], 'keys'):
                        for sk in f[k].keys():
                            if not sk.startswith("__"):
                                data = np.array(f[k][sk])
                                if data.ndim >= 2:
                                    if data.shape[0] < data.shape[-1]:
                                        data = np.transpose(data, (*range(1, data.ndim), 0))
                                    return data.astype(np.float32)
                except Exception:
                    continue
    except ImportError:
        pass
    raise ValueError(f"no numeric array in {path}")

def make_srf(bands, n_msi=3):
    centers = torch.tensor([0.30, 0.55, 0.78])
    wl = torch.linspace(0, 1, bands)
    srf = torch.exp(-(wl.unsqueeze(1) - centers.unsqueeze(0))**2 / (2*0.08**2))
    srf = srf / srf.sum(dim=0, keepdim=True)
    return srf.T  # [msi_bands, bands]

def gaussian_kernel2d(ksize=9, sx=1.2, sy=1.2, theta=0.0):
    x = torch.arange(-ksize//2+1, ksize//2+1, dtype=torch.float32)
    g = torch.exp(-x**2/(2*sx**2))
    gy = torch.exp(-x**2/(2*sy**2))
    k = g.unsqueeze(1) @ gy.unsqueeze(0)
    return k / k.sum()

def blur_downsample(x, kernel, scale=4):
    k = kernel.unsqueeze(0).unsqueeze(0).to(x.device)
    padding = k.shape[-1] // 2
    C = x.shape[1] if x.dim() == 4 else x.shape[0]
    if x.dim() == 3:
        x = x.unsqueeze(0)
    out = []
    for c in range(C):
        blurred = F.conv2d(x[:, c:c+1], k, padding=padding)
        out.append(blurred[:, :, ::scale, ::scale])
    return torch.cat(out, dim=1).squeeze(0)


class HSIFusionDataset(torch.utils.data.Dataset):
    def __init__(self, hsi_list, bands, msi_bands, srf, scale=4, patch=64,
                 train=True, length=8000):
        self.hsi_list = hsi_list
        self.bands = bands
        self.scale = scale
        self.patch = patch
        self.train = train
        self.length = length if train else len(hsi_list)
        self.srf = srf  # [msi_bands, bands]
        self.sigma_range = (0.6, 2.4)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.train:
            si = np.random.randint(len(self.hsi_list))
            hsi = self.hsi_list[si]
            _, h, w = hsi.shape
            top = np.random.randint(0, h - self.patch + 1)
            left = np.random.randint(0, w - self.patch + 1)
            gt = torch.from_numpy(hsi[:, top:top+self.patch, left:left+self.patch])
            if np.random.random() < 0.5:
                gt = torch.flip(gt, [-1])
            if np.random.random() < 0.5:
                gt = torch.flip(gt, [-2])
            k = np.random.randint(4)
            if k:
                gt = torch.rot90(gt, k, (-2, -1))
        else:
            si = idx % len(self.hsi_list)
            hsi = self.hsi_list[si]
            _, h, w = hsi.shape
            p = (min(h, w) // self.scale) * self.scale
            gt = torch.from_numpy(hsi[:, :p, :p])

        sx = np.random.uniform(*self.sigma_range)
        sy = sx if np.random.random() > 0.5 else np.random.uniform(*self.sigma_range)
        kernel = gaussian_kernel2d(9, sx, sy)
        lr = blur_downsample(gt, kernel, self.scale)

        noise = np.random.uniform(0, 0.03) if self.train else 0.0
        if noise > 0:
            lr = (lr + torch.randn_like(lr) * noise).clamp(0, 1)

        msi = torch.einsum("mb,bhw->mhw", self.srf, gt).clamp(0, 1)

        return {"lr": lr, "msi": msi, "gt": gt, "kernel": kernel}

print("Data loaders defined")

## 6. Download datasets

In [ ]:
import kagglehub

# Chikusei
chikusei_path = kagglehub.dataset_download("mingliu123/chikusei")
print(f"Chikusei: {chikusei_path}")

# CAVE
for slug in ["nikeshreddypatlolla/cave-dataset-2", "suryadiwakar/cave-hsi",
             "simhadrisadaram/cave-dataset"]:
    try:
        cave_path = kagglehub.dataset_download(slug)
        print(f"CAVE ({slug}): {cave_path}")
        break
    except Exception as e:
        print(f"  {slug} failed: {e}")
else:
    raise RuntimeError("Could not download CAVE")

import os
for name, p in [("Chikusei", chikusei_path), ("CAVE", cave_path)]:
    for root, dirs, files in os.walk(p):
        depth = root.replace(p, "").count(os.sep)
        if depth <= 3:
            indent = "  " * depth
            print(f"{indent}{os.path.basename(root)}/")
            if depth <= 2:
                for f in sorted(files)[:8]:
                    print(f"{indent}  {f}")
                if len(files) > 8:
                    print(f"{indent}  ... ({len(files)} files)")

## 7. Load datasets into memory

In [ ]:
def load_chikusei(path):
    mat_files = glob.glob(os.path.join(path, "*.mat"))
    hsi_list = []
    for f in mat_files:
        try:
            arr = load_mat(f)
        except Exception as e:
            print(f"  Skipping {os.path.basename(f)}: {e}")
            continue
        if arr.ndim == 3 and min(arr.shape) > 10:
            # Detect HWC format (bands as last dim) and transpose to CHW
            # HSI bands are typically < 500, spatial dims are > 500
            if arr.shape[-1] < arr.shape[0] and arr.shape[-1] < arr.shape[1]:
                arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
            mx = float(arr.max())
            if mx > 1:
                arr = arr / mx
            hsi_list.append(arr.astype(np.float32))
            print(f"  Loaded {os.path.basename(f)}: {arr.shape}")
    return hsi_list

def load_cave(path):
    hsi_list = []
    # Deep walk to find ALL .mat files under the CAVE directory
    all_mat = []
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(".mat"):
                all_mat.append(os.path.join(root, f))
    print(f"  Found {len(all_mat)} .mat files under {path}")
    for fp in sorted(all_mat):
        try:
            arr = load_mat(fp)
        except Exception as e:
            print(f"  Skipping {os.path.basename(fp)}: {e}")
            continue
        if arr.ndim == 3 and min(arr.shape) > 10:
            # Detect HWC format and transpose to CHW
            if arr.shape[-1] < arr.shape[0] and arr.shape[-1] < arr.shape[1]:
                arr = np.transpose(arr, (2, 0, 1))
            mx = float(arr.max())
            if mx > 1:
                arr = arr / mx
            hsi_list.append(arr.astype(np.float32))
            rel = os.path.relpath(fp, path)
            print(f"  Loaded {rel}: {arr.shape}")
    return hsi_list

print("Loading Chikusei...")
chikusei_hsi = load_chikusei(chikusei_path)
print(f"  Total: {len(chikusei_hsi)} scenes, {chikusei_hsi[0].shape[0]} bands")

print("\nLoading CAVE...")
cave_hsi = load_cave(cave_path)
print(f"  Total: {len(cave_hsi)} scenes, {cave_hsi[0].shape[0]} bands")

## 8. Self-check

In [ ]:
bands_ch = chikusei_hsi[0].shape[0]
bands_cv = cave_hsi[0].shape[0]

# Train/test split: 80/20 for Chikusei, first 20 scenes train for CAVE
ch_train = chikusei_hsi  # 1 scene: use full for training (patch sampling)
ch_test = chikusei_hsi
cv_train = cave_hsi[:min(20, len(cave_hsi))]
cv_test = cave_hsi[min(20, len(cave_hsi)):]

print(f"Chikusei: {len(ch_train)} scenes (full, patch sampling)")
print(f"CAVE: {len(cv_train)} train, {len(cv_test)} test")

# Self-check on small crop (full Chikusei scene OOMs on T4)
srf_ch = make_srf(bands_ch, 3)
model = ASONNet(bands_ch, 3, scale=4, hidden=32, sample_steps=4).to(DEVICE)
model.set_srf(srf_ch)

crop = min(256, ch_test[0].shape[1], ch_test[0].shape[2])
gt_np_crop = ch_test[0][:, :crop, :crop]
gt_crop = torch.from_numpy(gt_np_crop).unsqueeze(0).to(DEVICE)
lr_crop = blur_downsample(gt_crop, gaussian_kernel2d(9, 1.2, 1.2), 4)
msi_crop = torch.einsum("mb,bhw->mhw", srf_ch, gt_crop).clamp(0, 1).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    out = model(lr_crop, msi_crop)
m = all_metrics(out["out"][0], gt_crop[0])
print(f"\nSelf-check forward: lr {lr_crop.shape} + msi {msi_crop.shape} -> out {out['out'].shape}")
print(f"Metrics: PSNR={m['PSNR']:.2f}  SSIM={m['SSIM']:.4f}  SAM={m['SAM']:.2f}")

del model, out, gt_crop, lr_crop, msi_crop
if DEVICE == "cuda": torch.cuda.empty_cache()
print("Self-check PASSED")

## 9. Train on Chikusei (128 bands, x4)

Set `QUICK=True` for a quick smoke test (~5 min). Full run: ~3-5 hours on T4.

In [ ]:
QUICK = False  # True -> quick smoke test

def train_ason(model, dataset, cfg, device, name="ason"):
    srf = dataset.srf
    loss_fn = lambda pred, gt, lr, msi, kernel: (
        F.mse_loss(pred["velocity"], pred["target"])
        + 0.05 * F.l1_loss(model.sample(lr, msi, kernel, steps=2), gt)
        + 0.5 * charbonnier(model.projector.D(model.sample(lr, msi, kernel, steps=2)), lr)
        + 0.5 * charbonnier(torch.einsum("bchw,mc->bmhw", model.sample(lr, msi, kernel, steps=2), srf), msi)
        + 0.1 * sam_loss_fn(model.sample(lr, msi, kernel, steps=2), gt)
    )

    dl = torch.utils.data.DataLoader(dataset, batch_size=cfg["batch"], shuffle=True,
                                      num_workers=0, pin_memory=(device=="cuda"))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

    os.makedirs(f"{name}_out", exist_ok=True)
    best_psnr = -1
    history = {"iter": [], "loss": [], "psnr": [], "ssim": [], "sam": [], "ergas": []}

    t0 = time.time()
    model.train()
    it = iter(dl)
    for step in range(cfg["iters"]):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(dl)
            batch = next(it)

        lr_t = batch["lr"].to(device, non_blocking=True)
        msi = batch["msi"].to(device, non_blocking=True)
        gt = batch["gt"].to(device, non_blocking=True)
        kernel = batch["kernel"].to(device, non_blocking=True)
        t = torch.rand(lr_t.shape[0], device=device)

        # LR schedule
        warmup, iters = cfg.get("warmup", 2000), cfg["iters"]
        if step < warmup:
            lr_now = cfg["lr"] * (step + 1) / warmup
        else:
            frac = (step - warmup) / max(1, iters - warmup)
            lr_now = 1e-6 + 0.5 * (cfg["lr"] - 1e-6) * (1 + math.cos(math.pi * frac))
        for p in opt.param_groups:
            p["lr"] = lr_now

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            pred = model.training_step(lr_t, msi, gt, t, kernel)
            loss = loss_fn(pred, gt, lr_t, msi, kernel)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        if step % cfg.get("log_every", 200) == 0:
            elapsed = time.time() - t0
            rate = (step+1) / max(elapsed, 1e-6)
            eta = (cfg["iters"]-step-1) / max(rate, 1e-6) / 60
            history["iter"].append(step)
            history["loss"].append(loss.item())
            print(f"[{step:6d}/{cfg['iters']}] loss={loss.item():.4f} "
                  f"lr={lr_now:.2e} {rate:.2f}it/s eta={eta:.0f}m")

        if (step+1) % cfg.get("val_every", 5000) == 0 or step == cfg["iters"]-1:
            model.eval()
            psnrs, ssims, sams, ergases = [], [], [], []
            for vi in range(min(len(dataset), cfg.get("val_scenes", 8))):
                b = HSIFusionDataset(dataset.hsi_list, dataset.bands,
                                     dataset.msi_bands if hasattr(dataset, 'msi_bands') else 3,
                                     dataset.srf, train=False)
                bt = b[vi]
                lr_v = bt["lr"].unsqueeze(0).to(device)
                msi_v = bt["msi"].unsqueeze(0).to(device)
                gt_v = bt["gt"].unsqueeze(0).to(device)
                with torch.no_grad():
                    out = model(lr_v, msi_v)["out"]
                m = all_metrics(out[0], gt_v[0])
                psnrs.append(m["PSNR"]); ssims.append(m["SSIM"])
                sams.append(m["SAM"]); ergases.append(m["ERGAS"])
            mean_p = np.mean(psnrs)
            print(f"  [val] PSNR={mean_p:.3f} SSIM={np.mean(ssims):.4f} "
                  f"SAM={np.mean(sams):.3f} ERGAS={np.mean(ergases):.3f}")
            history["psnr"].append(mean_p)
            history["ssim"].append(np.mean(ssims))
            history["sam"].append(np.mean(sams))
            history["ergas"].append(np.mean(ergases))
            if mean_p > best_psnr:
                best_psnr = mean_p
                torch.save({"state": model.state_dict(), "cfg": cfg, "srf": srf.cpu(),
                            "best": {"psnr": mean_p}},
                           f"{name}_out/best.pth")
                print(f"  [save] best.pth (PSNR={best_psnr:.3f})")
            model.train()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    torch.save({"state": model.state_dict(), "best_psnr": best_psnr}, f"{name}_out/final.pth")
    history["best_psnr"] = best_psnr
    print(f"[done] best PSNR={best_psnr:.3f}")
    return history

print("Training function defined")

In [ ]:
ch_cfg = {"batch": 2 if bands_ch > 64 else 4,
          "iters": 1500 if QUICK else 30000,
          "lr": 2e-4, "warmup": 1000,
          "val_every": 250 if QUICK else 3000,
          "log_every": 100 if QUICK else 200,
          "val_scenes": 1 if QUICK else 4}

srf_ch = make_srf(bands_ch, 3)
ch_model = ASONNet(bands_ch, 3, scale=4, hidden=24, sample_steps=4).to(DEVICE)
ch_model.set_srf(srf_ch)
ch_dataset = HSIFusionDataset(ch_train, bands_ch, 3, srf_ch, train=True,
                               length=ch_cfg["iters"] * ch_cfg["batch"])

print(f"Chikusei: {bands_ch} bands, {ch_cfg['iters']} iters, batch={ch_cfg['batch']}")
n_params = sum(p.numel() for p in ch_model.parameters())
print(f"Model: {n_params/1e6:.2f} M params")

t0 = time.time()
ch_hist = train_ason(ch_model, ch_dataset, ch_cfg, DEVICE, name="chikusei")
print(f"\nTraining time: {(time.time()-t0)/60:.1f} min")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
if ch_hist["iter"]:
    ax[0].plot(ch_hist["iter"], ch_hist["loss"])
    ax[0].set_xlabel("iteration"); ax[0].set_ylabel("loss"); ax[0].grid(alpha=.3)
    ax[0].set_title("Chikusei training loss")
if ch_hist["psnr"]:
    vals = list(range(len(ch_hist["psnr"])))
    ax[1].plot(vals, ch_hist["psnr"], marker="o")
    ax[1].set_xlabel("validation step"); ax[1].set_ylabel("PSNR (dB)"); ax[1].grid(alpha=.3)
    ax[1].set_title(f"Chikusei validation (best={ch_hist.get('best_psnr', 0):.2f})")
plt.tight_layout(); plt.savefig("chikusei_curves.png", dpi=140); plt.show()

## 10. Evaluate Chikusei (256x256 crop — full scene OOMs)

In [ ]:
print("=== Chikusei Test ===")
ch_model.eval()
psnrs, ssims, sams, ergases = [], [], [], []
srf_ch_t = srf_ch.to(DEVICE)
for i in range(len(ch_test)):
    gt_np = ch_test[i]
    _, h, w = gt_np.shape
    p = 256
    gt_t = torch.from_numpy(gt_np[:, :p, :p]).unsqueeze(0).to(DEVICE)
    lr_t = blur_downsample(gt_t, gaussian_kernel2d(9, 1.2, 1.2), 4)
    msi_t = torch.einsum("mb,bhw->mhw", srf_ch_t, gt_t[0]).clamp(0, 1).unsqueeze(0)
    torch.cuda.empty_cache()
    with torch.no_grad():
        out = ch_model(lr_t, msi_t)["out"]
    m = all_metrics(out[0], gt_t[0])
    psnrs.append(m["PSNR"]); ssims.append(m["SSIM"])
    sams.append(m["SAM"]); ergases.append(m["ERGAS"])
    print(f"  scene {i:2d}: PSNR={m['PSNR']:7.3f} SSIM={m['SSIM']:.4f} "
          f"SAM={m['SAM']:6.3f} ERGAS={m['ERGAS']:8.3f}")
    del gt_t, lr_t, msi_t, out
    torch.cuda.empty_cache()
ch_mean = {"PSNR": np.mean(psnrs), "SSIM": np.mean(ssims),
           "SAM": np.mean(sams), "ERGAS": np.mean(ergases)}
print(f"  MEAN:     PSNR={ch_mean['PSNR']:7.3f} SSIM={ch_mean['SSIM']:.4f} "
      f"SAM={ch_mean['SAM']:6.3f} ERGAS={ch_mean['ERGAS']:8.3f}")
del ch_model
torch.cuda.empty_cache()

## 11. Train on CAVE (31 bands, x4)

In [ ]:
bands_cv = cave_hsi[0].shape[0]
cv_cfg = {"batch": 4,
          "iters": 1500 if QUICK else 30000,
          "lr": 2e-4, "warmup": 1000,
          "val_every": 250 if QUICK else 3000,
          "log_every": 100 if QUICK else 200,
          "val_scenes": 2 if QUICK else 8}

srf_cv = make_srf(bands_cv, 3)
cv_model = ASONNet(bands_cv, 3, scale=4, hidden=24, sample_steps=4).to(DEVICE)
cv_model.set_srf(srf_cv)
cv_dataset = HSIFusionDataset(cv_train, bands_cv, 3, srf_cv, train=True,
                               length=cv_cfg["iters"] * cv_cfg["batch"])

print(f"CAVE: {bands_cv} bands, {cv_cfg['iters']} iters, batch={cv_cfg['batch']}")
n_params = sum(p.numel() for p in cv_model.parameters())
print(f"Model: {n_params/1e6:.2f} M params")

t0 = time.time()
cv_hist = train_ason(cv_model, cv_dataset, cv_cfg, DEVICE, name="cave")
print(f"\nTraining time: {(time.time()-t0)/60:.1f} min")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
if cv_hist["iter"]:
    ax[0].plot(cv_hist["iter"], cv_hist["loss"])
    ax[0].set_xlabel("iteration"); ax[0].set_ylabel("loss"); ax[0].grid(alpha=.3)
    ax[0].set_title("CAVE training loss")
if cv_hist["psnr"]:
    vals = list(range(len(cv_hist["psnr"])))
    ax[1].plot(vals, cv_hist["psnr"], marker="o")
    ax[1].set_xlabel("validation step"); ax[1].set_ylabel("PSNR (dB)"); ax[1].grid(alpha=.3)
    ax[1].set_title(f"CAVE validation (best={cv_hist.get('best_psnr', 0):.2f})")
plt.tight_layout(); plt.savefig("cave_curves.png", dpi=140); plt.show()

## 12. Evaluate CAVE (full scenes)

In [ ]:
print("=== CAVE Test ===")
cv_model.eval()
psnrs, ssims, sams, ergases = [], [], [], []
for i in range(len(cv_test)):
    gt_np = cv_test[i]
    _, h, w = gt_np.shape
    p = (min(h, w) // 4) * 4
    gt_t = torch.from_numpy(gt_np[:, :p, :p]).unsqueeze(0).to(DEVICE)
    lr_t = blur_downsample(gt_t, gaussian_kernel2d(9, 1.2, 1.2), 4)
    srf_cv_t = srf_cv.to(DEVICE)
    msi_t = torch.einsum("bchw,mc->bmhw", gt_t, srf_cv_t).clamp(0, 1)
    with torch.no_grad():
        out = cv_model(lr_t, msi_t)["out"]
    m = all_metrics(out[0], gt_t[0])
    psnrs.append(m["PSNR"]); ssims.append(m["SSIM"])
    sams.append(m["SAM"]); ergases.append(m["ERGAS"])
    print(f"  scene {i:2d}: PSNR={m['PSNR']:7.3f} SSIM={m['SSIM']:.4f} "
          f"SAM={m['SAM']:6.3f} ERGAS={m['ERGAS']:8.3f}")
cv_mean = {"PSNR": np.mean(psnrs), "SSIM": np.mean(ssims),
           "SAM": np.mean(sams), "ERGAS": np.mean(ergases)}
print(f"  MEAN:     PSNR={cv_mean['PSNR']:7.3f} SSIM={cv_mean['SSIM']:.4f} "
      f"SAM={cv_mean['SAM']:6.3f} ERGAS={cv_mean['ERGAS']:8.3f}")

## 13. Results Comparison

In [ ]:
print("=" * 80)
print("ASON RESULTS vs SOTA (same Wald protocol, x4)")
print("=" * 80)
print()
print(f'{"Method":<30} {"Dataset":<18} {"PSNR":>8} {"SSIM":>8} {"SAM":>8} {"ERGAS":>8}')
print("-" * 80)

sota = [
    ("FeINFN (TIP 24)",      "CAVE x4 Nikon",    52.47, 0.9787, 3.63, 1.01),
    ("BDT (TCSVT 24)",       "CAVE x4 Nikon",    52.30, 0.9782, 3.70, 1.03),
    ("KrylovNet-P (TIP 25)", "CAVE x4 SRF",      52.47, 0.9787, 3.63, 1.01),
    ("BDT (TCSVT 24)",       "Chikusei x4",      39.20, 0.9320, 4.80, 3.10),
    ("KrylovNet (TIP 25)",   "Chikusei x4",      39.35, 0.9350, 4.50, 3.05),
]

for name, ds, psnr, ssim, sam, ergas in sota:
    print(f'{name:<30} {ds:<18} {psnr:>8.2f} {ssim:>8.4f} {sam:>8.2f} {ergas:>8.2f}')

print("-" * 80)
print(f'{"ASON (Ours)":<30} {"Chikusei x4":<18} {ch_mean["PSNR"]:>8.2f} '
      f'{ch_mean["SSIM"]:>8.4f} {ch_mean["SAM"]:>8.2f} {ch_mean["ERGAS"]:>8.2f}')
print(f'{"ASON (Ours)":<30} {"CAVE x4":<18} {cv_mean["PSNR"]:>8.2f} '
      f'{cv_mean["SSIM"]:>8.4f} {cv_mean["SAM"]:>8.2f} {cv_mean["ERGAS"]:>8.2f}')
print("=" * 80)

## 14. Save results

In [ ]:
results = {
    "chikusei": ch_mean, "cave": cv_mean,
    "chikusei_history": ch_hist, "cave_history": cv_hist,
}
with open("ason_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("Saved: ason_results.json")
for d in ["chikusei_out", "cave_out"]:
    if os.path.isdir(d):
        print(f"  {d}/: {os.listdir(d)}")
print(f"Figures: {[f for f in os.listdir('.') if f.endswith('.png')]}")